In [ ]:
"""
tests/test_pipeline.py - unit tests + end-to-end smoke test (pytest).
Customer Health & Sentiment Scoring Multi-Agent AI Solution

    pip install pytest
    pytest -q tests/test_pipeline.py                 # unit tests + smoke test (~1-2 min)
    pytest -q tests/test_pipeline.py -m "not smoke"  # unit tests only (seconds)

The smoke test generates synthetic raw CSVs in a temporary PIPELINE_ROOT and runs Steps 2 -> 3d through
run_pipeline.py in a subprocess (common.py resolves PIPELINE_ROOT at import time, so it needs a fresh process).
"""

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pytest

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import business_rules as br                           # noqa: E402
from cohort_scoring import max_mom_swing, score_candidates   # noqa: E402

## 2. business_rules - shared cohort rule SQL

In [ ]:
FLAGS = [(t, e, i) for t in (0, 1) for e in (0, 1) for i in (0, 1)]


def _eval(rule: str) -> dict:
    con = duckdb.connect()
    con.execute("CREATE TABLE f (t INT, e INT, i INT)")
    con.executemany("INSERT INTO f VALUES (?, ?, ?)", FLAGS)
    rows = con.execute(f"SELECT t, e, i, {br.core_source_sql('t', 'e', 'i', rule)} FROM f").fetchall()
    return {(t, e, i): ok for t, e, i, ok in rows}


def test_tx_plus_one():
    res = _eval("tx_plus_one")
    for (t, e, i), ok in res.items():
        assert ok == (t == 1 and (e == 1 or i == 1))


def test_any_two():
    res = _eval("any_two")
    for (t, e, i), ok in res.items():
        assert ok == (t + e + i >= 2)


def test_rule_works_on_booleans():
    """EDA passes BOOLEAN flags, production passes INT flags - same answer."""
    con = duckdb.connect()
    sql = br.core_source_sql("TRUE", "FALSE", "TRUE", "tx_plus_one")
    assert con.execute(f"SELECT {sql}").fetchone()[0] is True


def test_unknown_rule_rejected():
    with pytest.raises(ValueError):
        br.core_source_sql("t", "e", "i", "bogus")


def test_isin_empty_list_is_false():
    assert br.isin("x", ()) == "FALSE"


def test_ranges_consistent():
    assert br.RANGES["duration_sec"][1] >= 3_600, "interaction duration cap must be hours, not minutes"
    assert br.ENG_LONG_SESSION_WARN_SEC < br.RANGES["session_duration_sec"][1]

## 3. cohort_scoring - gate and ranking behaviour

In [ ]:
def _scorecard() -> pd.DataFrame:
    base = dict(coverage_pct=80.0, avg_active_months_history=6.0, survey_label_pct=20.0, interaction_label_pct=40.0,
                avg_spend_pctile=0.5, std_spend_pctile=0.3, risk_signal_pct=30.0, max_mom_swing_pct=5.0)
    rows = [dict(dimension="ALL", segment="ALL", n_core_customers=100_000, **base),
            dict(dimension="loyalty_tier", segment="gold", n_core_customers=30_000, **{**base, "coverage_pct": 90}),
            dict(dimension="loyalty_tier", segment="silver", n_core_customers=25_000, **base),
            dict(dimension="region", segment="tiny", n_core_customers=50, **{**base, "coverage_pct": 99})]
    return pd.DataFrame(rows)


def test_gate_and_order():
    out = score_candidates(_scorecard(), 20_000, br.COHORT_WEIGHTS)
    assert "ALL" not in set(out.dimension)
    assert out.iloc[0].segment == "gold"
    tiny = out[out.segment == "tiny"].iloc[0]
    assert tiny.cohort_score == 0 and not tiny.passes_sample_gate
    assert tiny.cohort_score_ungated > 0            # diagnostic score still available for EDA 12A


def test_tiny_segment_does_not_shift_ranks():
    with_tiny = score_candidates(_scorecard(), 20_000, br.COHORT_WEIGHTS)
    without = score_candidates(_scorecard().query("segment != 'tiny'"), 20_000, br.COHORT_WEIGHTS)
    a = with_tiny.set_index("segment").loc[["gold", "silver"], "cohort_score"]
    b = without.set_index("segment").loc[["gold", "silver"], "cohort_score"]
    assert a.equals(b)


def test_missing_column_raises():
    with pytest.raises(KeyError):
        score_candidates(_scorecard().drop(columns="coverage_pct"), 1, br.COHORT_WEIGHTS)


def test_max_mom_swing():
    m = pd.DataFrame({"segment": ["a"] * 3, "month": ["2026-01", "2026-02", "2026-03"], "n": [100, 150, 75]})
    assert max_mom_swing(m).max_mom_swing_pct.iloc[0] == pytest.approx(50.0)

## 4. End-to-end smoke test (Step 2 -> 3d on synthetic data)

In [ ]:
@pytest.mark.smoke
def test_end_to_end(tmp_path):
    env = {**os.environ, "PIPELINE_ROOT": str(tmp_path), "SMOKE_CUSTOMERS": "4000",
           "MIN_COHORT_CUSTOMERS": "300", "PIPELINE_THREADS": "2"}
    subprocess.run([sys.executable, str(ROOT / "tests" / "make_synthetic_bronze.py"), "--csv"],
                   check=True, env=env, cwd=ROOT)
    r = subprocess.run([sys.executable, str(ROOT / "run_pipeline.py")], env=env, cwd=ROOT,
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stdout[-3000:] + r.stderr[-3000:]

    cohort = tmp_path / "data" / "cohort" / "poc_customer_month.parquet"
    assert cohort.exists()
    con = duckdb.connect()
    n, dup, bad_hist, both = con.execute(f"""
        SELECT count(*), count(*) - count(DISTINCT (customer_id, month)),
               count(*) FILTER (WHERE months_observed < {br.MIN_HISTORY_MONTHS} OR active_m <> 1),
               (SELECT count(*) FROM (SELECT customer_id FROM '{cohort.as_posix()}'
                                      GROUP BY 1 HAVING count(DISTINCT split) > 1))
        FROM '{cohort.as_posix()}'""").fetchone()
    assert n > 0 and dup == 0 and bad_hist == 0 and both == 0
    cols = {c[0] for c in con.execute(f"DESCRIBE SELECT * FROM '{cohort.as_posix()}'").fetchall()}
    assert not cols & {"sentiment_label", "sentiment_score_gt", "interaction_text"}, "label leakage"
    used = json.loads((tmp_path / "reports" / "cohort" / "cohort_selection_used.json").read_text())
    assert used["status"] == "PASSED" and used["criteria"]["core_rule"] == br.CORE_RULE
    # D05: interaction durations are no longer wiped by a minutes-scale cap
    nulls = con.execute(f"""SELECT avg((duration_sec IS NULL)::INT)
                            FROM '{(tmp_path / "data" / "silver" / "interactions.parquet").as_posix()}'""").fetchone()[0]
    assert nulls < 0.01